# VQE and QAOA

Variational algorithms on exact simulations, compared with exact answers:
the full-CI energy of H₂, Lanczos ground energies of spin chains, and
brute-force maximum cuts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import metatron_qso as q

## H₂ in the STO-3G basis

Two qubits after the parity mapping with two-qubit reduction. VQE with a
one-layer RealAmplitudes ansatz reaches the full-CI energy.

In [ ]:
h2 = q.h2_sto3g()
result = q.run_vqe(h2["hamiltonian"], q.real_amplitudes(2, 1), restarts=4, seed=0)
energy = result["energy"] + h2["nuclear_repulsion"]
exact = h2["hamiltonian"].ground_state_energy() + h2["nuclear_repulsion"]
print(f"VQE   {energy:.10f} Ha")
print(f"exact {exact:.10f} Ha (full CI of the basis: -1.137306)")
print("iterations:", result["iterations"], "energies of the restarts:", np.round(result["restart_energies"], 8))

In [ ]:
plt.figure(figsize=(5, 3))
plt.semilogy(np.array(result["history"]) - h2["hamiltonian"].ground_state_energy() + 1e-16)
plt.xlabel("iteration")
plt.ylabel("energy error (Ha)")
plt.tight_layout()
plt.show()

## Depth matters: a transverse-field Ising chain

Six spins at the critical point J = h = 1. The error of VQE falls with the
number of ansatz layers.

In [ ]:
h = q.transverse_field_ising(6, 1.0, 1.0)
exact = h.ground_state_energy()
layers = [1, 2, 3, 4, 5, 6]
errors = []
for L in layers:
    r = q.run_vqe(h, q.efficient_su2(6, L), restarts=4, seed=1)
    errors.append(r["energy"] - exact)
    print(f"{L} layers: E = {r['energy']:.8f}, error {errors[-1]:.2e}")

plt.figure(figsize=(5, 3))
plt.semilogy(layers, np.maximum(errors, 1e-12), "o-")
plt.xlabel("layers")
plt.ylabel("E_VQE − E_0")
plt.tight_layout()
plt.show()

## QAOA for MaxCut

On a ring, depth-p QAOA reaches exactly (2p+1)/(2p+2) of the maximum cut.
Layer-wise optimisation (INTERP) carries each optimum to the next depth.

In [ ]:
graphs = {
    "ring C10": q.Graph.cycle(10),
    "Petersen": q.Graph.petersen(),
    "Metatron's Cube": q.Graph.metatron_cube(),
}
ratios = {}
for name, g in graphs.items():
    ratios[name] = [r["approximation_ratio"] for r in q.optimize_qaoa_layerwise(g, 4, seed=0)]
    print(name, np.round(ratios[name], 5))
print("ring, theory:", [round((2 * p + 1) / (2 * p + 2), 5) for p in range(1, 5)])

plt.figure(figsize=(5, 3))
for name, r in ratios.items():
    plt.plot(range(1, 5), r, "o-", label=name)
plt.xlabel("depth p")
plt.ylabel("approximation ratio")
plt.legend()
plt.tight_layout()
plt.show()

The most likely bit string of the optimised state is an optimal cut:

In [ ]:
qaoa = q.MaxCutQaoa(q.Graph.petersen(), 4)
best = qaoa.optimize(restarts=8, seed=0)
print(best["most_likely_bitstring"], "cut", best["most_likely_cut"], "of", qaoa.max_cut)
print("probability of an optimal cut:", round(best["optimal_probability"], 4))